# 04 — Future Melt Projection

Runs the iterative future melt simulation under three retreat rate scenarios (lower bound, central estimate, upper bound), using the MLP model's predicted melt probability on 2025 AlphaEarth embeddings to determine which pixels melt first at each decadal timestep.

All logic lives in `glacier_melt.projection`; this notebook only loads data, computes MLP scores, calls the simulation functions, and saves/plots results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset

from glacier_melt.sampling import load_simulation_parquets, AE_COLS
from glacier_melt.models import load_model
from glacier_melt.train import predict
from glacier_melt.projection import (
    prepare_simulation_pixels, run_all_scenarios, find_deglaciation_year,
    area_history_to_dataframe, SCENARIOS, TIMESTEPS,
)
from glacier_melt.visualise import plot_retreat_projection

SIM_DIR = Path("../data/Simulation_Data_2025")
MODEL_PATH = Path("../data/Final_Models/MLP_AE64_R3HOLDOUT.pt")
RGI_PATH = Path("../data/rgi70_peru.geojson")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(exist_ok=True)

## Load 2025 simulation pixels and match to RGI glaciers

In [2]:
df_sim = load_simulation_parquets(SIM_DIR)
df_matched = prepare_simulation_pixels(df_sim, RGI_PATH)

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

## Score pixels with the MLP model

Predicted melt probability on 2025 AlphaEarth embeddings serves as the
vulnerability score that determines melt order in the simulation.

In [ ]:
mlp_model = load_model(MODEL_PATH, "mlp", dropout=0.5)
device = next(mlp_model.parameters()).device

embeddings = torch.from_numpy(
    df_matched[AE_COLS].values.astype(np.float32)
).to(device)
dummy_labels = torch.zeros(len(df_matched), dtype=torch.float32, device=device)
scores = predict(mlp_model, TensorDataset(embeddings, dummy_labels))

print(f"Scores: min={scores.min():.4f}, mean={scores.mean():.4f}, "
      f"max={scores.max():.4f}")

## Run simulation under all three retreat rate scenarios

In [ ]:
start_area_km2 = len(df_matched) * 0.0001
print(f"Starting ice area (2025): {start_area_km2:.2f} km²")

scenario_results = run_all_scenarios(df_matched, scores, scenarios=SCENARIOS)

## Deglaciation year per scenario

In [ ]:
for name, rate in SCENARIOS.items():
    area_history, _ = scenario_results[name]
    deglac = find_deglaciation_year(area_history, rate)
    print(f"{name:>10} ({rate} km²/yr): deglaciation ~{deglac}")

## Save outputs

Saves the central scenario's per-pixel melt years (used by `05_figures.ipynb`
for the glacier projection panels) and the area history for all scenarios.

In [ ]:
_, central_melt_years = scenario_results["central"]
central_melt_years.to_parquet(
    OUTPUT_DIR / "simulation_melt_years.parquet", index=False
)

df_history = area_history_to_dataframe(scenario_results, start_area_km2)
df_history.to_parquet(OUTPUT_DIR / "simulation_area_history.parquet", index=False)

print("Saved simulation_melt_years.parquet and simulation_area_history.parquet")

## Retreat projection plot

In [ ]:
fig = plot_retreat_projection(start_area_km2, scenarios=SCENARIOS)
fig.savefig(OUTPUT_DIR / "glacier_retreat_projection.png", dpi=200, bbox_inches="tight")
plt.show()